<!-- field-paper-role-banner -->
> 🟦 **실습본** — TODO를 직접 구현하고 assertion을 통과시키세요. 정답본은 같은 위치의 필요한 부분만 확인합니다.

# 자연어 처리와 LLM · 05. BERT

## BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding

- **원 논문:** [BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding](https://arxiv.org/abs/1810.04805)
- **저자 / 발표:** Jacob Devlin, Ming-Wei Chang, Kenton Lee, Kristina Toutanova · NAACL 2019 (2018)
- **난이도 / 예상 시간:** 중급 · 약 120분
- **선수 지식:** Python, PyTorch, 확률·선형대수·sequence model의 기초
- **로컬 학습 데이터:** `data/field_curriculum/nlp_corpus.json`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** 80/10/10 MLM corruption과 bidirectional encoder를 재현한다.

**축소하거나 생략한 부분:** BERT Base/Large와 BooksCorpus+Wikipedia 대신 1 encoder block을 쓰고, 300개 train 행과 60개 held-out 행을 서로 다른 seed로 손상한다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | §3.1 Task #1: MLM and 80% mask/10% random/10% unchanged | Task 1 | train/evaluation별 selected labels와 정확한 80/10/10 corruption |
| Fig. 1 and §3 Fig. 2: bidirectional attention and three embeddings | Task 2 explicit unmasked Q/K/V attention | [B,H,T,T] positive attention weights |
| §3.1 Task #1: MLM objective | Task 3 | loss and masked-token accuracy |

## 핵심 아이디어와 수식

$$L_{MLM}=-\sum_{i\in\mathcal{M}}\log p(x_i\mid x_{\setminus\mathcal{M}})$$

**기호 해설:** M은 선택된 position, x_i는 원 token, 양쪽 context가 조건으로 사용된다.

**코드 대응:** Task 1이 80/10/10 corruption, Task 2가 mask 없는 Q/K/V 양방향 encoder, Task 3이 MLM CE를 구현한다.

**입력과 출력 shape:** train corruption [300,4], held-out corruption [60,4]이며, 10-row probe는 hidden [10,4,24]와 logits [10,4,V]를 만든다.

**포트폴리오 구현 범위:** NSP와 대규모 corpus는 제외하지만 masking 정책과 비인과적 self-attention은 보존한다.

각 단계는 데이터 전처리와 shape 확인부터 논문 고유 class, `forward`, 목적함수,
`train_step`, 평가까지 이어진다. 핵심 수식을 한 번의 고수준 호출로 감추지 않고
중간 tensor를 확인할 수 있게 분리한다.

In [ ]:
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
import json
import math
import re
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

torch.manual_seed(20260814)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device


def locate(relative_path):
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / relative_path
        if candidate.exists():
            return candidate
    raise FileNotFoundError(relative_path)


dataset_path = locate("data/field_curriculum/nlp_corpus.json")
corpus = json.loads(dataset_path.read_text(encoding="utf-8"))
sentences = corpus["sentences"]
translation_pairs = corpus["translation_pairs"]
documents = corpus["documents"]
queries = corpus["queries"]
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<mask>", "<sentinel>"]
sentence_words = {word.lower() for sentence in sentences for word in sentence.split()}
translation_words = {
    word.lower()
    for pair in translation_pairs
    for side in ("source", "target")
    for word in pair[side].split()
}
words = sorted(sentence_words | translation_words)
itos = SPECIAL_TOKENS + words
stoi = {word: index for index, word in enumerate(itos)}
PAD, BOS, EOS, MASK, SENTINEL = range(5)
VOCAB_SIZE = len(itos)


def encode(text, length=4):
    token_ids = [stoi[word.lower()] for word in text.split()]
    token_ids = token_ids[:length]
    return token_ids + [PAD] * (length - len(token_ids))


sentence_ids = torch.tensor([encode(text) for text in sentences])
sentiment_labels = torch.tensor(corpus["sentiment_labels"], dtype=torch.long)
source_ids = torch.tensor([encode(pair["source"], 3) for pair in translation_pairs])
target_ids = torch.tensor([encode(pair["target"], 3) for pair in translation_pairs])
sentence_ids, sentiment_labels = ACCELERATOR.move(
    sentence_ids,
    sentiment_labels,
)
source_ids, target_ids = ACCELERATOR.move(source_ids, target_ids)
assert sentence_ids.shape == (360, 4)
assert source_ids.shape == target_ids.shape == (60, 3)
print(ACCELERATOR.summary())
print(dataset_path.name, f"sentences={len(sentences)}, vocab={VOCAB_SIZE}")

### 구현 단계 1 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Task #1: MLM and 80% mask/10% random/10% unchanged
- **노트북에서 구현할 부분:** Task 1
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** train/evaluation별 selected labels와 정확한 80/10/10 corruption

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 1: BERT 설정과 선택 position에 8/1/1 corruption을 적용하세요.
@dataclass
class BERTConfig:
    pass


def apply_mlm_corruption(
    tokens,
    mask_id,
    random_token_ids,
    generator,
):
    raise NotImplementedError

### 구현 단계 2 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** Fig. 1 and §3 Fig. 2: bidirectional attention and three embeddings
- **노트북에서 구현할 부분:** Task 2 explicit unmasked Q/K/V attention
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** [B,H,T,T] positive attention weights

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 2: unmasked self-attention encoder와 MLM head를 갖는 BERTMini를 구현하세요.
class BidirectionalSelfAttention(nn.Module):
    """BERT Fig. 1의 mask 없는 multi-head self-attention이다."""

    def __init__(self, config):
        super().__init__()

    def split_heads(self, tensor):
        """[B, T, D]를 [B, H, T, d_h]로 분할한다."""
        raise NotImplementedError

    def merge_heads(self, tensor):
        """[B, H, T, d_h]를 [B, T, D]로 병합한다."""
        raise NotImplementedError

    def forward(self, hidden):
        """Mask 없는 attention output과 [B, H, T, T] weights를 반환한다."""
        raise NotImplementedError


class BERTMini(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()

    def embed(self, token_ids):
        """Token, position, segment embedding을 [B, T, D]로 합산한다."""
        raise NotImplementedError

    def forward(self, token_ids):
        raise NotImplementedError

### 구현 단계 3 — 원문 근거를 코드로 옮기기

- **논문의 어느 부분인가:** §3.1 Task #1: MLM objective
- **노트북에서 구현할 부분:** Task 3
- **구현 이유:** 원문의 확률 분해나 tensor 변환을 작은 입력에서 직접
  관찰하고, 고수준 API 뒤에 핵심 알고리즘을 숨기지 않기 위해서다.
- **완료 증거:** loss and masked-token accuracy

아래 TODO 전에 입력 token/feature shape와 수식의 축을 먼저 확인한다.

In [ ]:
# TODO 3: selected-position MLM loss, train step, MLM accuracy를 구현하세요.
def masked_language_model_loss(logits, labels):
    raise NotImplementedError


def train_bert_step(model, optimizer, corrupted, labels):
    raise NotImplementedError


def evaluate_mlm(model, corrupted, labels):
    raise NotImplementedError

## 실험 기록

1. 핵심 수식의 기호와 코드 변수 이름을 대응시키세요.
2. 어떤 assertion이 가장 중요한 구현 오류를 잡는지 설명하세요.
3. 이 미니 데이터로 검증할 수 없는 원 논문의 주장을 하나 적으세요.
4. seed를 유지하고 변수 하나만 바꾼 뒤 metric 전후를 기록하세요.